# Experiment: Manifold Structure with Correlated Pairs

This notebook tests whether nonlinear encoders develop manifold structure when trained
on distributions with **correlated feature pairs**.

Using `CorrelatedPairs`, features are organized in pairs (2i, 2i+1) where both features
in a pair tend to activate together. This creates a natural context-dependence structure
that the encoder must handle.

**Key questions:**
1. Does feature correlation promote manifold structure (context-dependent encoding directions)?
2. Do paired features show different encoding behavior than unpaired baselines?
3. How does correlation strength affect manifold structure emergence?

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch import Generator

from occhio import ToyModel, MLPAutoencoder
from occhio.autoencoder import TiedLinear, TiedLinearRelu
from occhio.distributions.sparse import SparseUniform
from occhio.distributions.correlated import CorrelatedPairs
from occhio.analysis import (
    compute_feature_jacobians,
    angular_variance,
    jacobian_pca,
    direction_vs_context,
)

## 1. Configuration

- n=200 features (100 pairs), m=20 hidden (10:1 compression)
- Correlated pairs with configurable correlation strength
- Each pair activates with p_active, then individual features activate with p_individual

In [ ]:
# Configuration
N_FEATURES = 200  # Must be even for pairs
N_PAIRS = N_FEATURES // 2
N_HIDDEN = 20  # 10:1 compression ratio
N_EPOCHS = 15000
BATCH_SIZE = 512
N_JACOBIAN_SAMPLES = 1000

# CorrelatedPairs parameters
# Using correlation + density parameterization for intuitive control
CORRELATION = 0.7  # Correlation between paired features
DENSITY = 0.03  # Effective density (p_active * p_individual)

print(f"Configuration: {N_FEATURES} features ({N_PAIRS} pairs) -> {N_HIDDEN} hidden")
print(f"Compression ratio: {N_FEATURES / N_HIDDEN:.1f}:1")
print(f"Target correlation: {CORRELATION}")
print(f"Target density: {DENSITY}")
print(f"Expected active features per sample: {N_FEATURES * DENSITY:.1f}")

In [ ]:
# Visualize the correlation structure by sampling
test_dist = CorrelatedPairs(
    N_FEATURES,
    correlation=CORRELATION,
    density=DENSITY,
    generator=Generator().manual_seed(42),
)
test_samples = test_dist.sample(10000)

# Compute empirical statistics
empirical_density = (test_samples > 0).float().mean().item()
print(f"Empirical density: {empirical_density:.4f}")

# Compute correlation between first few pairs
pair_correlations = []
for i in range(min(10, N_PAIRS)):
    feat_a = test_samples[:, 2 * i]
    feat_b = test_samples[:, 2 * i + 1]
    # Binary correlation (both active)
    a_active = (feat_a > 0).float()
    b_active = (feat_b > 0).float()
    corr = torch.corrcoef(torch.stack([a_active, b_active]))[0, 1].item()
    pair_correlations.append(corr)

print(f"Mean empirical pair correlation: {np.mean(pair_correlations):.4f}")
print(f"Individual pair correlations: {[f'{c:.3f}' for c in pair_correlations[:5]]}")

# Visualize co-activation patterns
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Pair Co-activation", "Non-pair Co-activation"]
)

# Pair co-activation (should be high)
pair_both_active = []
for i in range(N_PAIRS):
    both = (
        ((test_samples[:, 2 * i] > 0) & (test_samples[:, 2 * i + 1] > 0))
        .float()
        .mean()
        .item()
    )
    pair_both_active.append(both)

fig.add_trace(
    go.Histogram(x=pair_both_active, nbinsx=30, name="Paired features"), row=1, col=1
)

# Non-pair co-activation (should be lower, just chance)
non_pair_both_active = []
for i in range(min(100, N_PAIRS)):
    # Compare feature 2i with feature 2j+1 where i != j
    j = (i + 1) % N_PAIRS
    both = (
        ((test_samples[:, 2 * i] > 0) & (test_samples[:, 2 * j + 1] > 0))
        .float()
        .mean()
        .item()
    )
    non_pair_both_active.append(both)

fig.add_trace(
    go.Histogram(x=non_pair_both_active, nbinsx=30, name="Non-paired features"),
    row=1,
    col=2,
)

fig.update_layout(
    title=f"Co-activation Rates (correlation={CORRELATION}, density={DENSITY})",
    height=400,
)
fig.show()

print(f"\nMean co-activation rate:")
print(f"  Paired features: {np.mean(pair_both_active):.4f}")
print(f"  Non-paired features: {np.mean(non_pair_both_active):.4f}")
print(f"  Ratio: {np.mean(pair_both_active) / np.mean(non_pair_both_active):.2f}x")

## 2. Train Models

Training three architectures on CorrelatedPairs:
1. **TiedLinear**: Linear baseline (angular variance should be ~0)
2. **TiedLinearRelu**: Piecewise linear (ReLU decoder)
3. **MLPAutoencoder**: Smooth nonlinearity (GELU encoder)

Plus a comparison with uncorrelated SparseUniform.

In [ ]:
def create_correlated_distribution(seed=42):
    """Create CorrelatedPairs distribution."""
    return CorrelatedPairs(
        N_FEATURES,
        correlation=CORRELATION,
        density=DENSITY,
        generator=Generator().manual_seed(seed),
    )


def create_uncorrelated_distribution(seed=42):
    """Create SparseUniform with same density but no correlation (baseline)."""
    return SparseUniform(
        n_features=N_FEATURES,
        p_active=DENSITY,
        generator=Generator().manual_seed(seed),
    )


# Verify distribution samples
dist = create_correlated_distribution()
samples = dist.sample(1000)
print(f"Sample shape: {samples.shape}")
print(f"Feature 0 active rate: {(samples[:, 0] > 0).float().mean():.3f}")
print(f"Feature 1 active rate: {(samples[:, 1] > 0).float().mean():.3f}")
print(
    f"Features 0 & 1 both active: {((samples[:, 0] > 0) & (samples[:, 1] > 0)).float().mean():.3f}"
)
print(f"Mean active features per sample: {(samples > 0).sum(dim=1).float().mean():.1f}")

In [ ]:
# Train TiedLinear (linear baseline) on correlated pairs
print("Training TiedLinear on CorrelatedPairs...")
linear_model = ToyModel(
    distribution=create_correlated_distribution(),
    ae=TiedLinear(n_features=N_FEATURES, n_hidden=N_HIDDEN),
)
linear_losses, _ = linear_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {linear_losses[-1]:.6f}")

In [ ]:
# Train TiedLinearRelu (piecewise linear) on correlated pairs
print("Training TiedLinearRelu on CorrelatedPairs...")
relu_model = ToyModel(
    distribution=create_correlated_distribution(),
    ae=TiedLinearRelu(n_features=N_FEATURES, n_hidden=N_HIDDEN),
)
relu_losses, _ = relu_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {relu_losses[-1]:.6f}")

In [ ]:
# Train MLPAutoencoder (smooth nonlinearity) on correlated pairs
print("Training MLPAutoencoder on CorrelatedPairs...")
mlp_model = ToyModel(
    distribution=create_correlated_distribution(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN,
        encoder_hidden_dim=N_HIDDEN * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
)
mlp_losses, _ = mlp_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_losses[-1]:.6f}")

In [ ]:
# Train MLP on UNCORRELATED distribution for comparison
print("Training MLPAutoencoder on SparseUniform (uncorrelated comparison)...")
mlp_uncorr_model = ToyModel(
    distribution=create_uncorrelated_distribution(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN,
        encoder_hidden_dim=N_HIDDEN * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
)
mlp_uncorr_losses, _ = mlp_uncorr_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_uncorr_losses[-1]:.6f}")

In [ ]:
# Plot training losses
fig = go.Figure()
fig.add_trace(go.Scatter(y=linear_losses, name="TiedLinear (Correlated)", mode="lines"))
fig.add_trace(
    go.Scatter(y=relu_losses, name="TiedLinearRelu (Correlated)", mode="lines")
)
fig.add_trace(go.Scatter(y=mlp_losses, name="MLP (Correlated)", mode="lines"))
fig.add_trace(
    go.Scatter(
        y=mlp_uncorr_losses,
        name="MLP (Uncorrelated)",
        mode="lines",
        line=dict(dash="dash"),
    )
)
fig.update_layout(
    title="Training Loss Comparison",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    yaxis_type="log",
    height=400,
)
fig.show()

## 3. Experiment A: Does Manifold Structure Emerge?

Compute angular variance for all features across models.
If MLP develops significantly higher AV than linear baseline, manifold structure is present.

In [ ]:
# Generate test samples for Jacobian analysis
test_dist = create_correlated_distribution(seed=999)
test_samples = test_dist.sample(N_JACOBIAN_SAMPLES * 2)

# For uncorrelated comparison
test_dist_uncorr = create_uncorrelated_distribution(seed=999)
test_samples_uncorr = test_dist_uncorr.sample(N_JACOBIAN_SAMPLES * 2)

print(f"Test samples shape: {test_samples.shape}")

In [ ]:
def compute_all_angular_variances(model, samples, n_samples_per_feature=500):
    """Compute angular variance for all features."""
    n_features = model.n_features
    avs = []

    for feat_idx in range(n_features):
        # Filter for samples where this feature is active
        active_mask = samples[:, feat_idx] > 0
        active_samples = samples[active_mask]

        if len(active_samples) < 10:
            avs.append(np.nan)
            continue

        active_samples = active_samples[:n_samples_per_feature]

        jacs = compute_feature_jacobians(model, feat_idx, active_samples)
        av = angular_variance(jacs)
        avs.append(av)

        if feat_idx % 50 == 0:
            print(f"  Feature {feat_idx}: AV = {av:.6f} (n={len(active_samples)})")

    return np.array(avs)


print("Computing angular variance for all features...")
print("\nTiedLinear (Correlated):")
linear_avs = compute_all_angular_variances(linear_model, test_samples)

print("\nTiedLinearRelu (Correlated):")
relu_avs = compute_all_angular_variances(relu_model, test_samples)

print("\nMLPAutoencoder (Correlated):")
mlp_avs = compute_all_angular_variances(mlp_model, test_samples)

print("\nMLPAutoencoder (Uncorrelated):")
mlp_uncorr_avs = compute_all_angular_variances(mlp_uncorr_model, test_samples_uncorr)

In [ ]:
# Summary statistics (excluding NaN)
def summarize_avs(avs, name):
    valid = avs[~np.isnan(avs)]
    print(f"{name}:")
    print(f"  Mean AV: {valid.mean():.6f}")
    print(f"  Max AV:  {valid.max():.6f}")
    print(f"  Std AV:  {valid.std():.6f}")
    print(f"  Valid features: {len(valid)}/{len(avs)}")
    return valid


print("=" * 60)
print("ANGULAR VARIANCE SUMMARY")
print("=" * 60)
linear_valid = summarize_avs(linear_avs, "TiedLinear (Correlated)")
print()
relu_valid = summarize_avs(relu_avs, "TiedLinearRelu (Correlated)")
print()
mlp_valid = summarize_avs(mlp_avs, "MLPAutoencoder (Correlated)")
print()
mlp_uncorr_valid = summarize_avs(mlp_uncorr_avs, "MLPAutoencoder (Uncorrelated)")

In [ ]:
# Plot angular variance comparison
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Angular Variance by Feature (Correlated Pairs)",
        "Distribution Comparison",
        "AV: Primary vs Secondary Features",
        "Correlated vs Uncorrelated MLP",
    ],
    specs=[[{}, {}], [{}, {}]],
)

features = list(range(N_FEATURES))

# Per-feature AV (Correlated models)
fig.add_trace(
    go.Scatter(x=features, y=linear_avs, name="TiedLinear", mode="lines", opacity=0.7),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=features, y=relu_avs, name="TiedLinearRelu", mode="lines", opacity=0.7
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=features, y=mlp_avs, name="MLP (Correlated)", mode="lines"),
    row=1,
    col=1,
)

# Box plot comparison
fig.add_trace(
    go.Box(y=linear_valid, name="TiedLinear", boxpoints="outliers"), row=1, col=2
)
fig.add_trace(
    go.Box(y=relu_valid, name="TiedLinearRelu", boxpoints="outliers"), row=1, col=2
)
fig.add_trace(
    go.Box(y=mlp_valid, name="MLP (Correlated)", boxpoints="outliers"), row=1, col=2
)

# AV: Primary (even indices) vs Secondary (odd indices) features
primary_avs = mlp_avs[0::2]  # Features 0, 2, 4, ...
secondary_avs = mlp_avs[1::2]  # Features 1, 3, 5, ...
fig.add_trace(
    go.Box(
        y=primary_avs[~np.isnan(primary_avs)], name="Primary (2i)", boxpoints="outliers"
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Box(
        y=secondary_avs[~np.isnan(secondary_avs)],
        name="Secondary (2i+1)",
        boxpoints="outliers",
    ),
    row=2,
    col=1,
)

# Correlated vs Uncorrelated MLP comparison
fig.add_trace(
    go.Box(y=mlp_valid, name="MLP Correlated", boxpoints="outliers"), row=2, col=2
)
fig.add_trace(
    go.Box(y=mlp_uncorr_valid, name="MLP Uncorrelated", boxpoints="outliers"),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=2)
fig.update_yaxes(title_text="Angular Variance", row=2, col=1)
fig.update_yaxes(title_text="Angular Variance", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Angular Variance Analysis: Correlated Pairs",
    showlegend=True,
)
fig.show()

In [ ]:
# Key analysis: Does paired partner's activity predict direction change?
# For each feature, compare its AV when computed from samples where:
# 1. The paired partner IS active
# 2. The paired partner is NOT active


def compute_conditional_av(model, feat_idx, samples, condition_idx, n_samples=300):
    """Compute AV for feat_idx conditional on condition_idx being active or not."""
    # Samples where feat_idx is active
    feat_active = samples[:, feat_idx] > 0
    cond_active = samples[:, condition_idx] > 0

    # Both active
    both_active_samples = samples[feat_active & cond_active][:n_samples]
    # feat active, partner not active
    feat_only_samples = samples[feat_active & ~cond_active][:n_samples]

    if len(both_active_samples) < 10 or len(feat_only_samples) < 10:
        return np.nan, np.nan, 0, 0

    jacs_both = compute_feature_jacobians(model, feat_idx, both_active_samples)
    jacs_alone = compute_feature_jacobians(model, feat_idx, feat_only_samples)

    av_both = angular_variance(jacs_both)
    av_alone = angular_variance(jacs_alone)

    return av_both, av_alone, len(both_active_samples), len(feat_only_samples)


# Compute conditional AV for all pairs
print("Computing conditional angular variance (partner active vs not)...")
pair_analysis = []

for pair_idx in range(N_PAIRS):
    primary_idx = 2 * pair_idx
    secondary_idx = 2 * pair_idx + 1

    # Primary feature conditioned on secondary
    av_both_p, av_alone_p, n_both_p, n_alone_p = compute_conditional_av(
        mlp_model, primary_idx, test_samples, secondary_idx
    )

    # Secondary feature conditioned on primary
    av_both_s, av_alone_s, n_both_s, n_alone_s = compute_conditional_av(
        mlp_model, secondary_idx, test_samples, primary_idx
    )

    pair_analysis.append(
        {
            "pair": pair_idx,
            "primary_av_with_partner": av_both_p,
            "primary_av_without_partner": av_alone_p,
            "secondary_av_with_partner": av_both_s,
            "secondary_av_without_partner": av_alone_s,
            "n_both_primary": n_both_p,
            "n_alone_primary": n_alone_p,
        }
    )

    if pair_idx % 20 == 0:
        print(
            f"  Pair {pair_idx}: Primary AV with/without = {av_both_p:.4f}/{av_alone_p:.4f}"
        )

# Summarize
valid_pairs = [p for p in pair_analysis if not np.isnan(p["primary_av_with_partner"])]
print(f"\nValid pairs for analysis: {len(valid_pairs)}/{N_PAIRS}")

## 4. Partner-Conditioned Analysis

Key question: Does the presence/absence of the paired partner systematically affect
the encoding direction? If so, the MLP is using context-dependent encoding.

In [ ]:
# Analyze: Is AV lower when conditioned on a specific context (partner active)?
# Hypothesis: If the encoding direction depends on partner, then:
# - Overall AV (mixed samples) should be high
# - Conditional AV (partner active OR not) should be lower

primary_av_with = [p["primary_av_with_partner"] for p in valid_pairs]
primary_av_without = [p["primary_av_without_partner"] for p in valid_pairs]
secondary_av_with = [p["secondary_av_with_partner"] for p in valid_pairs]
secondary_av_without = [p["secondary_av_without_partner"] for p in valid_pairs]

# Compute overall AV for comparison
primary_overall_av = [mlp_avs[2 * p["pair"]] for p in valid_pairs]
secondary_overall_av = [mlp_avs[2 * p["pair"] + 1] for p in valid_pairs]

print("=" * 60)
print("CONDITIONAL ANGULAR VARIANCE ANALYSIS")
print("=" * 60)
print("\nPrimary features (2i):")
print(f"  Overall AV: {np.nanmean(primary_overall_av):.4f}")
print(f"  AV when partner active: {np.nanmean(primary_av_with):.4f}")
print(f"  AV when partner inactive: {np.nanmean(primary_av_without):.4f}")

print("\nSecondary features (2i+1):")
print(f"  Overall AV: {np.nanmean(secondary_overall_av):.4f}")
print(f"  AV when partner active: {np.nanmean(secondary_av_with):.4f}")
print(f"  AV when partner inactive: {np.nanmean(secondary_av_without):.4f}")

# Statistical test: Is conditional AV lower than overall AV?
from scipy import stats

t_stat, p_val = stats.ttest_rel(
    [p for p in primary_av_with if not np.isnan(p)],
    [p for p in primary_overall_av if not np.isnan(p)],
)
print(
    f"\nPaired t-test (conditional vs overall for primary): t={t_stat:.3f}, p={p_val:.4e}"
)

In [ ]:
# Visualize conditional AV analysis
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Primary Feature: Overall vs Conditional AV",
        "Secondary Feature: Overall vs Conditional AV",
        "AV With vs Without Partner (Primary)",
        "AV Reduction by Conditioning",
    ],
)

# Box plots for primary features
fig.add_trace(
    go.Box(y=primary_overall_av, name="Overall", marker_color="blue"), row=1, col=1
)
fig.add_trace(
    go.Box(y=primary_av_with, name="Partner Active", marker_color="green"), row=1, col=1
)
fig.add_trace(
    go.Box(y=primary_av_without, name="Partner Inactive", marker_color="red"),
    row=1,
    col=1,
)

# Box plots for secondary features
fig.add_trace(
    go.Box(y=secondary_overall_av, name="Overall", marker_color="blue"), row=1, col=2
)
fig.add_trace(
    go.Box(y=secondary_av_with, name="Partner Active", marker_color="green"),
    row=1,
    col=2,
)
fig.add_trace(
    go.Box(y=secondary_av_without, name="Partner Inactive", marker_color="red"),
    row=1,
    col=2,
)

# Scatter: AV with partner vs without partner
fig.add_trace(
    go.Scatter(
        x=primary_av_with,
        y=primary_av_without,
        mode="markers",
        marker=dict(size=5, opacity=0.6),
        name="Primary",
        showlegend=False,
    ),
    row=2,
    col=1,
)
# Diagonal line (equal AV)
max_val = max(max(primary_av_with), max(primary_av_without))
fig.add_trace(
    go.Scatter(
        x=[0, max_val],
        y=[0, max_val],
        mode="lines",
        line=dict(dash="dash", color="gray"),
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Histogram of AV reduction (overall - conditional)
av_reduction_with = np.array(primary_overall_av) - np.array(primary_av_with)
av_reduction_without = np.array(primary_overall_av) - np.array(primary_av_without)
fig.add_trace(
    go.Histogram(x=av_reduction_with, name="Reduction (partner active)", opacity=0.6),
    row=2,
    col=2,
)
fig.add_trace(
    go.Histogram(
        x=av_reduction_without, name="Reduction (partner inactive)", opacity=0.6
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="AV (partner active)", row=2, col=1)
fig.update_yaxes(title_text="AV (partner inactive)", row=2, col=1)
fig.update_xaxes(title_text="AV Reduction from Overall", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Conditional Angular Variance: Does Partner Context Matter?",
    barmode="overlay",
)
fig.show()

# Summary statistics
print("\nAV reduction when conditioning on partner status:")
print(f"  Mean reduction (partner active): {np.nanmean(av_reduction_with):.4f}")
print(f"  Mean reduction (partner inactive): {np.nanmean(av_reduction_without):.4f}")
print(
    f"  Features with positive reduction (active): {np.sum(av_reduction_with > 0)}/{len(av_reduction_with)}"
)
print(
    f"  Features with positive reduction (inactive): {np.sum(av_reduction_without > 0)}/{len(av_reduction_without)}"
)

In [ ]:
# Direction separation analysis: Do Jacobians for "partner active" vs "partner inactive"
# point in systematically different directions?


def compute_mean_direction_separation(
    model, feat_idx, samples, partner_idx, n_samples=200
):
    """Compute angular separation between mean Jacobian directions for partner active vs not."""
    feat_active = samples[:, feat_idx] > 0
    partner_active = samples[:, partner_idx] > 0

    both_active = samples[feat_active & partner_active][:n_samples]
    feat_only = samples[feat_active & ~partner_active][:n_samples]

    if len(both_active) < 10 or len(feat_only) < 10:
        return np.nan

    jacs_both = compute_feature_jacobians(model, feat_idx, both_active)
    jacs_alone = compute_feature_jacobians(model, feat_idx, feat_only)

    # Mean directions (normalized)
    mean_both = jacs_both.mean(dim=0)
    mean_alone = jacs_alone.mean(dim=0)

    mean_both = mean_both / mean_both.norm().clamp(min=1e-8)
    mean_alone = mean_alone / mean_alone.norm().clamp(min=1e-8)

    # Cosine similarity -> angular separation
    cos_sim = (mean_both @ mean_alone).item()
    cos_sim = np.clip(cos_sim, -1, 1)
    angle_rad = np.arccos(cos_sim)
    angle_deg = np.degrees(angle_rad)

    return angle_deg


# Compute direction separation for all pairs
print("Computing mean direction separation (partner active vs inactive)...")
direction_separations = []
for pair_idx in range(N_PAIRS):
    primary_idx = 2 * pair_idx
    secondary_idx = 2 * pair_idx + 1

    sep = compute_mean_direction_separation(
        mlp_model, primary_idx, test_samples, secondary_idx
    )
    direction_separations.append(sep)

    if pair_idx % 20 == 0 and not np.isnan(sep):
        print(f"  Pair {pair_idx}: {sep:.1f} degrees")

valid_seps = [s for s in direction_separations if not np.isnan(s)]
print(f"\nMean direction separation: {np.mean(valid_seps):.1f} degrees")
print(f"Max separation: {np.max(valid_seps):.1f} degrees")
print(f"Min separation: {np.min(valid_seps):.1f} degrees")

## 5. Direction Separation Analysis

For features where the paired partner can be present or absent, do the encoding directions
systematically differ? This tests whether the MLP learns to use different encoding
strategies based on whether correlated features co-occur.

In [ ]:
# Visualize direction separations
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Distribution of Direction Separations",
        "Direction Separation vs Overall AV",
    ],
)

# Histogram of separations
fig.add_trace(
    go.Histogram(x=valid_seps, nbinsx=30, name="Direction Separation"), row=1, col=1
)

# Scatter: separation vs overall AV
valid_sep_arr = np.array(direction_separations)
valid_mask = ~np.isnan(valid_sep_arr)
primary_avs_for_scatter = np.array([mlp_avs[2 * i] for i in range(N_PAIRS)])

fig.add_trace(
    go.Scatter(
        x=valid_sep_arr[valid_mask],
        y=primary_avs_for_scatter[valid_mask],
        mode="markers",
        marker=dict(size=6, opacity=0.6),
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Direction Separation (degrees)", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_xaxes(title_text="Direction Separation (degrees)", row=1, col=2)
fig.update_yaxes(title_text="Overall Angular Variance", row=1, col=2)

fig.update_layout(height=400, title_text="Partner-Conditioned Direction Analysis")
fig.show()

# Correlation between separation and AV
from scipy import stats as sp_stats

valid_seps_arr = valid_sep_arr[valid_mask]
valid_primary_avs = primary_avs_for_scatter[valid_mask]
corr, pval = sp_stats.pearsonr(valid_seps_arr, valid_primary_avs)
print(f"\nCorrelation (Direction Separation vs Overall AV): r={corr:.4f}, p={pval:.4e}")

if corr > 0.3 and pval < 0.05:
    print(
        "=> POSITIVE correlation: Features with larger partner-dependent direction shifts have higher AV"
    )
    print("   This confirms that AV captures context-dependent encoding!")
else:
    print("=> No strong correlation between direction separation and overall AV")

In [ ]:
# Compare to linear model (should have zero separation since it's context-independent)
print("Computing direction separations for LINEAR model (should be ~0)...")
linear_separations = []
for pair_idx in range(min(20, N_PAIRS)):  # Just sample a few
    primary_idx = 2 * pair_idx
    secondary_idx = 2 * pair_idx + 1

    sep = compute_mean_direction_separation(
        linear_model, primary_idx, test_samples, secondary_idx
    )
    if not np.isnan(sep):
        linear_separations.append(sep)

print(
    f"\nLinear model mean direction separation: {np.mean(linear_separations):.2f} degrees"
)
print(f"MLP model mean direction separation: {np.mean(valid_seps):.2f} degrees")
print(
    f"\nRatio (MLP/Linear): {np.mean(valid_seps) / max(np.mean(linear_separations), 0.01):.1f}x"
)

# Collect AVs for each PAIR (matching separations)
primary_avs = []
secondary_avs = []
for pair_idx in range(N_PAIRS):
    primary_avs.append(mlp_avs[2 * pair_idx])
    secondary_avs.append(mlp_avs[2 * pair_idx + 1])

## 6. 3D Visualization (Low-Dimensional)

Train models with n_hidden=3 to directly visualize how paired feature encoding
directions differ based on partner presence.

In [ ]:
# Smaller models for 3D visualization
N_FEATURES_3D = 50
N_PAIRS_3D = N_FEATURES_3D // 2
N_HIDDEN_3D = 3
N_EPOCHS_3D = 5000


def create_correlated_distribution_3d(seed=42):
    return CorrelatedPairs(
        N_FEATURES_3D,
        correlation=CORRELATION,
        density=DENSITY,
        generator=Generator().manual_seed(seed),
    )


print("Training 3D models...")

linear_3d = ToyModel(
    distribution=create_correlated_distribution_3d(),
    ae=TiedLinear(n_features=N_FEATURES_3D, n_hidden=N_HIDDEN_3D),
)
linear_3d.fit(n_epochs=N_EPOCHS_3D, batch_size=BATCH_SIZE)
print("Linear 3D trained.")

mlp_3d = ToyModel(
    distribution=create_correlated_distribution_3d(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES_3D,
        n_hidden=N_HIDDEN_3D,
        encoder_hidden_dim=N_HIDDEN_3D * 4,
        activation="gelu",
        decoder_activation="relu",
    ),
)
mlp_3d.fit(n_epochs=N_EPOCHS_3D, batch_size=BATCH_SIZE)
print("MLP 3D trained.")

In [ ]:
# Generate test samples for 3D models
test_dist_3d = create_correlated_distribution_3d(seed=999)
test_samples_3d = test_dist_3d.sample(3000)

# Find a pair with enough samples in both conditions
best_pair = None
best_count = 0
for pair_idx in range(N_PAIRS_3D):
    primary_idx = 2 * pair_idx
    secondary_idx = 2 * pair_idx + 1

    primary_active = test_samples_3d[:, primary_idx] > 0
    secondary_active = test_samples_3d[:, secondary_idx] > 0

    n_both = (primary_active & secondary_active).sum().item()
    n_primary_only = (primary_active & ~secondary_active).sum().item()

    if min(n_both, n_primary_only) > best_count:
        best_count = min(n_both, n_primary_only)
        best_pair = pair_idx

print(
    f"Best pair for visualization: {best_pair} with {best_count} samples in each condition"
)

# Get samples for this pair
primary_idx = 2 * best_pair
secondary_idx = 2 * best_pair + 1

primary_active = test_samples_3d[:, primary_idx] > 0
secondary_active = test_samples_3d[:, secondary_idx] > 0

samples_both = test_samples_3d[primary_active & secondary_active][:200]
samples_primary_only = test_samples_3d[primary_active & ~secondary_active][:200]

print(f"Samples with both active: {len(samples_both)}")
print(f"Samples with primary only: {len(samples_primary_only)}")

In [ ]:
# Compute Jacobians for both conditions
jacs_both = compute_feature_jacobians(mlp_3d, primary_idx, samples_both)
jacs_primary_only = compute_feature_jacobians(mlp_3d, primary_idx, samples_primary_only)

# Normalize to unit sphere
jacs_both_normed = jacs_both / jacs_both.norm(dim=1, keepdim=True).clamp(min=1e-8)
jacs_primary_normed = jacs_primary_only / jacs_primary_only.norm(
    dim=1, keepdim=True
).clamp(min=1e-8)

# Also compute for linear model
linear_jacs_both = compute_feature_jacobians(linear_3d, primary_idx, samples_both)
linear_jacs_primary = compute_feature_jacobians(
    linear_3d, primary_idx, samples_primary_only
)
linear_both_normed = linear_jacs_both / linear_jacs_both.norm(
    dim=1, keepdim=True
).clamp(min=1e-8)
linear_primary_normed = linear_jacs_primary / linear_jacs_primary.norm(
    dim=1, keepdim=True
).clamp(min=1e-8)

# 3D scatter plot
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[
        f"Linear Model (Feature {primary_idx})",
        f"MLP Model (Feature {primary_idx})",
    ],
)

# Linear model
fig.add_trace(
    go.Scatter3d(
        x=linear_both_normed[:, 0].detach().cpu().numpy(),
        y=linear_both_normed[:, 1].detach().cpu().numpy(),
        z=linear_both_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(size=3, color="green", opacity=0.6),
        name="Partner Active",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter3d(
        x=linear_primary_normed[:, 0].detach().cpu().numpy(),
        y=linear_primary_normed[:, 1].detach().cpu().numpy(),
        z=linear_primary_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(size=3, color="red", opacity=0.6),
        name="Partner Inactive",
    ),
    row=1,
    col=1,
)

# MLP model
fig.add_trace(
    go.Scatter3d(
        x=jacs_both_normed[:, 0].detach().cpu().numpy(),
        y=jacs_both_normed[:, 1].detach().cpu().numpy(),
        z=jacs_both_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(size=3, color="green", opacity=0.6),
        name="Partner Active",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter3d(
        x=jacs_primary_normed[:, 0].detach().cpu().numpy(),
        y=jacs_primary_normed[:, 1].detach().cpu().numpy(),
        z=jacs_primary_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(size=3, color="red", opacity=0.6),
        name="Partner Inactive",
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=f"Jacobian Directions: Partner Active (green) vs Inactive (red)",
    height=500,
)
fig.show()

# Compute mean directions and separation
mean_both = jacs_both_normed.mean(dim=0)
mean_both = mean_both / mean_both.norm()
mean_primary = jacs_primary_normed.mean(dim=0)
mean_primary = mean_primary / mean_primary.norm()
cos_sim = (mean_both @ mean_primary).item()
sep_deg = np.degrees(np.arccos(np.clip(cos_sim, -1, 1)))

print(f"\nMLP direction separation: {sep_deg:.1f} degrees")

# Linear
mean_both_lin = linear_both_normed.mean(dim=0)
mean_both_lin = mean_both_lin / mean_both_lin.norm()
mean_primary_lin = linear_primary_normed.mean(dim=0)
mean_primary_lin = mean_primary_lin / mean_primary_lin.norm()
cos_sim_lin = (mean_both_lin @ mean_primary_lin).item()
sep_deg_lin = np.degrees(np.arccos(np.clip(cos_sim_lin, -1, 1)))

print(f"Linear direction separation: {sep_deg_lin:.1f} degrees")

## 7. Reconstruction Quality Comparison

Compare reconstruction loss between correlated and uncorrelated distributions,
and between linear and MLP architectures.

In [ ]:
# Compute reconstruction loss on held-out data
test_dist_eval = create_correlated_distribution(seed=12345)
eval_samples = test_dist_eval.sample(5000)


def compute_reconstruction_loss(model, samples):
    with torch.no_grad():
        result = model.ae(samples)
        reconstructed = result[0] if isinstance(result, tuple) else result
        mse = ((samples - reconstructed) ** 2).mean().item()
    return mse


linear_loss = compute_reconstruction_loss(linear_model, eval_samples)
relu_loss = compute_reconstruction_loss(relu_model, eval_samples)
mlp_loss = compute_reconstruction_loss(mlp_model, eval_samples)

# Uncorrelated comparison
eval_samples_uncorr = create_uncorrelated_distribution(seed=12345).sample(5000)
mlp_uncorr_loss = compute_reconstruction_loss(mlp_uncorr_model, eval_samples_uncorr)

print("=" * 60)
print("RECONSTRUCTION LOSS COMPARISON (held-out data)")
print("=" * 60)
print(f"TiedLinear (Correlated):      {linear_loss:.6f}")
print(f"TiedLinearRelu (Correlated):  {relu_loss:.6f}")
print(f"MLPAutoencoder (Correlated):  {mlp_loss:.6f}")
print(f"MLPAutoencoder (Uncorrelated): {mlp_uncorr_loss:.6f}")
print()
print(
    f"MLP vs Linear improvement (Correlated): {(linear_loss - mlp_loss) / linear_loss * 100:.2f}%"
)
print()
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Compression ratio: {N_FEATURES}:{N_HIDDEN} = {N_FEATURES / N_HIDDEN:.1f}:1")
print(f"Correlation: {CORRELATION}")
print(f"Density: {DENSITY}")
print(f"Expected active features per sample: {N_FEATURES * DENSITY:.1f}")

## 8. Correlation Strength Sweep

How does the strength of correlation affect manifold structure?
Test multiple correlation values to see if stronger correlations lead to more
context-dependent encoding.

In [ ]:
# Test different correlation strengths
CORRELATION_VALUES = [0.0, 0.3, 0.5, 0.7, 0.9]
N_EPOCHS_SWEEP = 8000

correlation_results = {}

for corr in CORRELATION_VALUES:
    print(f"\n{'=' * 60}")
    print(f"Testing correlation = {corr}")
    print(f"{'=' * 60}")

    # Create distribution with this correlation
    def make_dist(seed=42):
        if corr == 0:
            # Use SparseUniform for zero correlation
            return SparseUniform(
                N_FEATURES, p_active=DENSITY, generator=Generator().manual_seed(seed)
            )
        return CorrelatedPairs(
            N_FEATURES,
            correlation=corr,
            density=DENSITY,
            generator=Generator().manual_seed(seed),
        )

    # Train MLP
    mlp = ToyModel(
        distribution=make_dist(),
        ae=MLPAutoencoder(
            n_features=N_FEATURES,
            n_hidden=N_HIDDEN,
            encoder_hidden_dim=N_HIDDEN * 2,
            activation="gelu",
            decoder_activation="relu",
        ),
    )
    mlp.fit(n_epochs=N_EPOCHS_SWEEP, batch_size=BATCH_SIZE)

    # Compute mean AV
    test_data = make_dist(seed=999).sample(2000)
    avs = []
    for feat_idx in range(0, N_FEATURES, 5):  # Sample every 5th feature
        active_mask = test_data[:, feat_idx] > 0
        active_samples = test_data[active_mask][:200]
        if len(active_samples) >= 10:
            jacs = compute_feature_jacobians(mlp, feat_idx, active_samples)
            avs.append(angular_variance(jacs))

    mean_av = np.mean(avs)

    # Compute reconstruction loss
    eval_data = make_dist(seed=54321).sample(5000)
    mse = compute_reconstruction_loss(mlp, eval_data)

    correlation_results[corr] = {
        "mean_av": mean_av,
        "mse": mse,
    }

    print(f"  Mean AV: {mean_av:.4f}")
    print(f"  MSE: {mse:.6f}")

In [ ]:
# Correlation between angular variance and reconstruction improvement
# This is the key test: if high-AV features show better reconstruction improvement,
# manifold structure is doing useful work

from scipy import stats


# Compute per-feature reconstruction loss for both models
def compute_per_feature_loss(model, samples):
    """Compute MSE per feature."""
    with torch.no_grad():
        result = model.ae(samples)
        reconstructed = result[0] if isinstance(result, tuple) else result
        per_feat_mse = ((samples - reconstructed) ** 2).mean(dim=0)
    return per_feat_mse.cpu().numpy()


linear_per_feat = compute_per_feature_loss(linear_model, eval_samples)
mlp_per_feat = compute_per_feature_loss(mlp_model, eval_samples)

# Calculate per-feature improvement and feature importance
improvement_per_feat = ((linear_per_feat - mlp_per_feat) / linear_per_feat) * 100
# Filter out NaN angular variances
valid_av_mask = ~np.isnan(mlp_avs)
valid_avs = mlp_avs[valid_av_mask]
valid_improvements = improvement_per_feat[valid_av_mask]

correlation, p_value = stats.pearsonr(valid_avs, valid_improvements)
spearman_corr, spearman_p = stats.spearmanr(valid_avs, valid_improvements)

print("=" * 60)
print("CORRELATION: Angular Variance vs Reconstruction Improvement")
print("=" * 60)
print(f"Pearson correlation:  r = {correlation:.4f}, p = {p_value:.4e}")
print(
    f"Spearman correlation: ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ = {spearman_corr:.4f}, p = {spearman_p:.4e}"
)
print()

if correlation > 0.2 and p_value < 0.05:
    print("=> POSITIVE correlation: High-AV features show better reconstruction!")
    print("   Manifold structure is doing useful work.")
elif correlation < -0.2 and p_value < 0.05:
    print("=> NEGATIVE correlation: High-AV features reconstruct worse.")
    print("   Manifold structure may be parasitic.")
else:
    print("=> No significant correlation detected.")
    print("   MLP improvement is spread across features regardless of AV.")

In [ ]:
# Calculate feature importance (mean squared value when active)
IMPORTANCES = torch.zeros(N_FEATURES)
for feat_idx in range(N_FEATURES):
    active_mask = eval_samples[:, feat_idx] > 0
    if active_mask.sum() > 0:
        IMPORTANCES[feat_idx] = (eval_samples[active_mask, feat_idx] ** 2).mean()
    else:
        IMPORTANCES[feat_idx] = 0.0
IMPORTANCES = IMPORTANCES.clamp(min=1e-10)  # Avoid log(0)

# Visualize per-feature reconstruction and AV correlation
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Per-Feature MSE: Linear vs MLP",
        "MLP Improvement by Feature",
        "Angular Variance vs Reconstruction Improvement",
        "Improvement vs Feature Importance",
    ],
)

# Per-feature MSE comparison
fig.add_trace(
    go.Scatter(
        x=list(range(N_FEATURES)),
        y=linear_per_feat,
        name="Linear",
        mode="lines",
        opacity=0.7,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=list(range(N_FEATURES)), y=mlp_per_feat, name="MLP", mode="lines"),
    row=1,
    col=1,
)

# Improvement per feature
colors = ["green" if x > 0 else "red" for x in improvement_per_feat]
fig.add_trace(
    go.Bar(
        x=list(range(N_FEATURES)),
        y=improvement_per_feat,
        marker_color=colors,
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

# AV vs Improvement scatter (the key plot)
fig.add_trace(
    go.Scatter(
        x=valid_avs,
        y=valid_improvements,
        mode="markers",
        marker=dict(
            size=6,
            color=IMPORTANCES.numpy()[valid_av_mask],
            colorscale="Viridis",
            colorbar=dict(title="Importance", x=1.15),
        ),
        showlegend=False,
        text=[f"Feature {i}" for i in np.where(valid_av_mask)[0]],
        hovertemplate="Feature %{text}<br>AV: %{x:.4f}<br>Improvement: %{y:.2f}%<extra></extra>",
    ),
    row=2,
    col=1,
)
# Add trendline
z = np.polyfit(valid_avs, valid_improvements, 1)
p = np.poly1d(z)
x_trend = np.linspace(valid_avs.min(), valid_avs.max(), 100)
fig.add_trace(
    go.Scatter(
        x=x_trend,
        y=p(x_trend),
        mode="lines",
        line=dict(dash="dash", color="red"),
        name=f"Trend (r={correlation:.2f})",
        showlegend=True,
    ),
    row=2,
    col=1,
)

# Improvement vs Importance
fig.add_trace(
    go.Scatter(
        x=IMPORTANCES.numpy(),
        y=improvement_per_feat,
        mode="markers",
        marker=dict(size=6, color=mlp_avs, colorscale="Plasma"),
        showlegend=False,
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="MSE", row=1, col=1)
fig.update_xaxes(title_text="Feature Index", row=1, col=2)
fig.update_yaxes(title_text="Improvement (%)", row=1, col=2)
fig.update_xaxes(title_text="Angular Variance", row=2, col=1)
fig.update_yaxes(title_text="Reconstruction Improvement (%)", row=2, col=1)
fig.update_xaxes(title_text="Feature Importance", type="log", row=2, col=2)
fig.update_yaxes(title_text="Reconstruction Improvement (%)", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Per-Feature Reconstruction Analysis: Does AV Predict Improvement?",
)
fig.show()

In [ ]:
print("=" * 70)
print("EXPERIMENT SUMMARY: Manifold Structure with Correlated Pairs")
print("=" * 70)
print()
print("CONFIGURATION:")
print(f"  Compression ratio: {N_FEATURES}:{N_HIDDEN} = {N_FEATURES / N_HIDDEN:.0f}:1")
print(f"  Correlation: {CORRELATION}")
print(f"  Density: {DENSITY}")
print(f"  Expected active features/sample: {N_FEATURES * DENSITY:.1f}")
print()

print("1. ANGULAR VARIANCE (manifold structure detection):")
print(f"   - Linear baseline: {np.nanmean(linear_avs):.6f} (expected ~0)")
print(f"   - MLP (Correlated): {np.nanmean(mlp_avs):.6f}")
print(f"   - MLP (Uncorrelated): {np.nanmean(mlp_uncorr_avs):.6f}")
print()

av_increase = np.nanmean(mlp_avs) / max(np.nanmean(linear_avs), 1e-10)
if av_increase > 10:
    print("   => SIGNIFICANT manifold structure detected in MLP!")
elif av_increase > 2:
    print("   => Moderate manifold structure detected.")
else:
    print("   => MLP converged to near-linear encoding.")

print()
print("2. PARTNER-CONDITIONED ANALYSIS:")
print(
    f"   - Mean direction separation (partner active vs not): {np.mean(valid_seps):.1f} degrees"
)
print(f"   - Linear model separation: {np.mean(linear_separations):.1f} degrees")

if np.mean(valid_seps) > 5:
    print("   => MLP uses DIFFERENT encoding directions depending on partner presence!")
else:
    print("   => MLP does not strongly differentiate based on partner presence.")

print()
print("3. RECONSTRUCTION QUALITY:")
print(f"   - Linear (Correlated): {linear_loss:.6f}")
print(f"   - MLP (Correlated): {mlp_loss:.6f}")
improvement = (linear_loss - mlp_loss) / linear_loss * 100
print(f"   - Improvement: {improvement:.2f}%")

print()
print("4. CORRELATION STRENGTH SWEEP:")
if correlation_results:
    for c in sorted(correlation_results.keys()):
        res = correlation_results[c]
        print(
            f"   - Correlation {c}: Mean AV = {res['mean_av']:.4f}, MSE = {res['mse']:.6f}"
        )

print()
print("=" * 70)
print("KEY INSIGHTS")
print("=" * 70)
print("""
This experiment tests whether CORRELATED PAIRS promote manifold structure:

1. When features are correlated (tend to co-activate), the MLP can potentially
   learn context-dependent encoding: different directions depending on whether
   the partner is present.

2. The DIRECTION SEPARATION metric measures how much the encoding direction
   changes based on partner presence. High separation = context-dependent.

3. CONDITIONAL ANGULAR VARIANCE: If overall AV is high but conditional AV
   (given partner state) is low, the MLP is using partner information to
   select different encoding directions.

4. The CORRELATION SWEEP shows how manifold structure changes with correlation
   strength. Stronger correlations may promote more context-dependent encoding.
""")

In [ ]:
import numpy as np
from scipy import stats

# Analyze per-pair separation distribution
separations = np.array(direction_separations)
separations = separations[~np.isnan(separations)]  # Remove NaN values

print(f"Mean separation: {np.mean(separations):.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿")
print(f"Std separation: {np.std(separations):.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿")
print(
    f"Min: {np.min(separations):.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿, Max: {np.max(separations):.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿"
)
print(f"Median: {np.median(separations):.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿")

# Quartiles
q25, q50, q75 = np.percentile(separations, [25, 50, 75])
print(
    f"Quartiles: Q1={q25:.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿, Q2={q50:.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿, Q3={q75:.2f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿"
)

# Identify pairs with large vs small separation
large_sep_mask = separations > np.mean(separations) + np.std(separations)
small_sep_mask = separations < np.mean(separations) - np.std(separations)

print(
    f"\nPairs with large separation (>1﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿): {np.sum(large_sep_mask)} pairs"
)
print(
    f"Pairs with small separation (<1﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿): {np.sum(small_sep_mask)} pairs"
)

# Compare angular variance for high vs low separation pairs
# Filter to match valid separations
valid_sep_indices = [i for i, s in enumerate(direction_separations) if not np.isnan(s)]
primary_avs_arr = np.array([primary_avs[i] for i in valid_sep_indices])
secondary_avs_arr = np.array([secondary_avs[i] for i in valid_sep_indices])

if np.sum(large_sep_mask) > 0 and np.sum(small_sep_mask) > 0:
    print(
        f"\nHigh-sep pairs - mean primary AV: {np.mean(primary_avs_arr[large_sep_mask]):.4f}"
    )
    print(
        f"Low-sep pairs - mean primary AV: {np.mean(primary_avs_arr[small_sep_mask]):.4f}"
    )

In [ ]:
# Analysis: Do high-separation pairs have higher interference with OTHER pairs?
# This tests whether "neighborhood pressure" from the broader population drives separation

print("=" * 70)
print("INTERFERENCE WITH BROADER POPULATION (not partner)")
print("=" * 70)

# Get the weight matrix W from the linear model (decoder weights)
W = linear_model.ae.W.detach().cpu().numpy()  # Shape: (n_hidden, n_features)


# Compute interference: |W_i ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ W_j| for each pair of features (normalized)
def compute_non_partner_interference(W, pair_idx):
    """
    Compute mean interference of a pair's features with ALL OTHER features
    (excluding their own partner).
    """
    primary_idx = 2 * pair_idx
    secondary_idx = 2 * pair_idx + 1

    # Get normalized feature vectors
    w_primary = W[:, primary_idx]
    w_secondary = W[:, secondary_idx]
    w_primary = w_primary / (np.linalg.norm(w_primary) + 1e-10)
    w_secondary = w_secondary / (np.linalg.norm(w_secondary) + 1e-10)

    interferences_primary = []
    interferences_secondary = []

    for other_pair in range(N_PAIRS):
        if other_pair == pair_idx:
            continue

        other_primary = 2 * other_pair
        other_secondary = 2 * other_pair + 1

        for other_idx in [other_primary, other_secondary]:
            w_other = W[:, other_idx]
            w_other = w_other / (np.linalg.norm(w_other) + 1e-10)

            interferences_primary.append(np.abs(w_primary @ w_other))
            interferences_secondary.append(np.abs(w_secondary @ w_other))

    return np.mean(interferences_primary), np.mean(interferences_secondary)


# Compute non-partner interference for all pairs
non_partner_interference = []
for pair_idx in valid_sep_indices:
    int_p, int_s = compute_non_partner_interference(W, pair_idx)
    non_partner_interference.append((int_p + int_s) / 2)

non_partner_interference = np.array(non_partner_interference)

# Correlate with separation
corr_interf_sep, p_interf_sep = stats.pearsonr(non_partner_interference, separations)
print(f"\nCorrelation (non-partner interference vs separation):")
print(f"  Pearson r = {corr_interf_sep:.4f}, p = {p_interf_sep:.4e}")

# Compare high vs low separation pairs
high_sep_interference = non_partner_interference[large_sep_mask]
low_sep_interference = non_partner_interference[small_sep_mask]

print(f"\nMean interference with other pairs:")
print(f"  High-separation pairs: {np.mean(high_sep_interference):.4f}")
print(f"  Low-separation pairs:  {np.mean(low_sep_interference):.4f}")

t_stat_interf, p_val_interf = stats.ttest_ind(
    high_sep_interference, low_sep_interference
)
print(f"  t-test: t={t_stat_interf:.3f}, p={p_val_interf:.4f}")

if corr_interf_sep > 0.2 and p_interf_sep < 0.05:
    print("\n=> HIGH-SEPARATION pairs have MORE interference with broader population")
    print("   This supports the 'neighborhood pressure' hypothesis!")
elif corr_interf_sep < -0.2 and p_interf_sep < 0.05:
    print("\n=> HIGH-SEPARATION pairs have LESS interference with broader population")
else:
    print("\n=> No clear relationship between separation and neighborhood interference")

# ============================================================================
# IMPORTANCE CORRELATION
# ============================================================================
print("\n" + "=" * 70)
print("IMPORTANCE CORRELATION WITH SEPARATION")
print("=" * 70)

# Get importance values for primary features in each pair
importance_per_pair = []
for pair_idx in valid_sep_indices:
    primary_idx = 2 * pair_idx
    importance_per_pair.append(IMPORTANCES[primary_idx].item())

importance_per_pair = np.array(importance_per_pair)

# Correlate importance with separation
corr_imp_sep, p_imp_sep = stats.pearsonr(importance_per_pair, separations)
spearman_imp_sep, spearman_p_imp_sep = stats.spearmanr(importance_per_pair, separations)

print(f"\nCorrelation (importance vs separation):")
print(f"  Pearson r = {corr_imp_sep:.4f}, p = {p_imp_sep:.4e}")
print(
    f"  Spearman ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ = {spearman_imp_sep:.4f}, p = {spearman_p_imp_sep:.4e}"
)

# Compare importance for high vs low separation pairs
high_sep_importance = importance_per_pair[large_sep_mask]
low_sep_importance = importance_per_pair[small_sep_mask]

print(f"\nMean importance:")
print(f"  High-separation pairs: {np.mean(high_sep_importance):.4f}")
print(f"  Low-separation pairs:  {np.mean(low_sep_importance):.4f}")

t_stat_imp, p_val_imp = stats.ttest_ind(high_sep_importance, low_sep_importance)
print(f"  t-test: t={t_stat_imp:.3f}, p={p_val_imp:.4f}")

# ============================================================================
# VISUALIZATION
# ============================================================================
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Separation vs Non-Partner Interference",
        "Separation vs Feature Importance",
        "Interference Distribution by Separation Group",
        "Importance Distribution by Separation Group",
    ],
)

# Separation vs Interference
fig.add_trace(
    go.Scatter(
        x=non_partner_interference,
        y=separations,
        mode="markers",
        marker=dict(size=6, color=importance_per_pair, colorscale="Viridis"),
        text=[f"Pair {i}" for i in valid_sep_indices],
        hovertemplate="Pair %{text}<br>Interference: %{x:.4f}<br>Separation: %{y:.1f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)
# Trendline
z_interf = np.polyfit(non_partner_interference, separations, 1)
p_interf = np.poly1d(z_interf)
x_interf = np.linspace(
    non_partner_interference.min(), non_partner_interference.max(), 50
)
fig.add_trace(
    go.Scatter(
        x=x_interf,
        y=p_interf(x_interf),
        mode="lines",
        line=dict(dash="dash", color="red"),
        name=f"r={corr_interf_sep:.2f}",
    ),
    row=1,
    col=1,
)

# Separation vs Importance
fig.add_trace(
    go.Scatter(
        x=importance_per_pair,
        y=separations,
        mode="markers",
        marker=dict(size=6, color=non_partner_interference, colorscale="Plasma"),
        text=[f"Pair {i}" for i in valid_sep_indices],
        hovertemplate="Pair %{text}<br>Importance: %{x:.4f}<br>Separation: %{y:.1f}﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=2,
)
z_imp = np.polyfit(importance_per_pair, separations, 1)
p_imp = np.poly1d(z_imp)
x_imp = np.linspace(importance_per_pair.min(), importance_per_pair.max(), 50)
fig.add_trace(
    go.Scatter(
        x=x_imp,
        y=p_imp(x_imp),
        mode="lines",
        line=dict(dash="dash", color="red"),
        name=f"r={corr_imp_sep:.2f}",
        showlegend=False,
    ),
    row=1,
    col=2,
)

# Box plots: Interference by separation group
fig.add_trace(
    go.Box(y=low_sep_interference, name="Low Sep", marker_color="blue"), row=2, col=1
)
fig.add_trace(
    go.Box(y=high_sep_interference, name="High Sep", marker_color="orange"),
    row=2,
    col=1,
)

# Box plots: Importance by separation group
fig.add_trace(
    go.Box(y=low_sep_importance, name="Low Sep", marker_color="blue", showlegend=False),
    row=2,
    col=2,
)
fig.add_trace(
    go.Box(
        y=high_sep_importance, name="High Sep", marker_color="orange", showlegend=False
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="Non-Partner Interference", row=1, col=1)
fig.update_yaxes(title_text="Direction Separation (﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿)", row=1, col=1)
fig.update_xaxes(title_text="Feature Importance", row=1, col=2)
fig.update_yaxes(title_text="Direction Separation (﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿)", row=1, col=2)
fig.update_yaxes(title_text="Non-Partner Interference", row=2, col=1)
fig.update_yaxes(title_text="Feature Importance", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Do High-Separation Pairs Face More Neighborhood Pressure?",
)
fig.show()

# Summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"""
Neighborhood Pressure Hypothesis:
  - Correlation (interference vs separation): r = {corr_interf_sep:.3f}
  - {"SUPPORTED" if corr_interf_sep > 0.2 and p_interf_sep < 0.05 else "NOT SUPPORTED"}: High-separation pairs {"DO" if corr_interf_sep > 0.2 else "do NOT"} have more interference

Importance Correlation:
  - Correlation (importance vs separation): r = {corr_imp_sep:.3f}
  - {"More important features" if corr_imp_sep > 0.2 else "Less important features" if corr_imp_sep < -0.2 else "Importance does NOT predict"} {"show larger separation" if abs(corr_imp_sep) > 0.2 else "separation"}
""")

In [ ]:
# ============================================================================
# FAIR COMPARISON: Give MLP the same inductive bias as TiedLinearRelu
# ============================================================================
# The TiedLinearRelu benefits from non-negative reconstruction matching
# non-negative features. The MLP with decoder_activation="relu" already does this,
# but let's verify and also test clamping to [0,1] to match data bounds exactly.

print("=" * 70)
print("FAIR COMPARISON: MLP with matched inductive biases")
print("=" * 70)

# Check: what fraction of MLP outputs are negative (should be ~0 with relu)?
with torch.no_grad():
    mlp_out, _ = mlp_model.ae(eval_samples)
    negative_frac = (mlp_out < 0).float().mean().item()
    over_one_frac = (mlp_out > 1).float().mean().item()
    print(f"\nCurrent MLP (decoder_activation='relu'):")
    print(f"  Fraction of outputs < 0: {negative_frac:.4%}")
    print(f"  Fraction of outputs > 1: {over_one_frac:.4%}")

# Also check linear model
with torch.no_grad():
    linear_out = linear_model.ae(eval_samples)[0]
    linear_neg_frac = (linear_out < 0).float().mean().item()
    linear_over_frac = (linear_out > 1).float().mean().item()
    print(f"\nLinear model (no activation):")
    print(f"  Fraction of outputs < 0: {linear_neg_frac:.4%}")
    print(f"  Fraction of outputs > 1: {linear_over_frac:.4%}")

# Check TiedLinearRelu
with torch.no_grad():
    relu_out = relu_model.ae(eval_samples)[0]
    relu_neg_frac = (relu_out < 0).float().mean().item()
    relu_over_frac = (relu_out > 1).float().mean().item()
    print(f"\nTiedLinearRelu:")
    print(f"  Fraction of outputs < 0: {relu_neg_frac:.4%}")
    print(f"  Fraction of outputs > 1: {relu_over_frac:.4%}")

# ============================================================================
# Train a new MLP with output clamped to [0, 1] for fairer comparison
# ============================================================================
print("\n" + "=" * 70)
print("Training MLP with clamped output [0, 1]...")
print("=" * 70)


class MLPAutoencoderClamped(MLPAutoencoder):
    """MLP with output clamped to [0, 1] to match data bounds."""

    def decode(self, z):
        out = super().decode(z)
        return out.clamp(0, 1)


mlp_clamped_model = ToyModel(
    distribution=create_correlated_distribution(),
    ae=MLPAutoencoderClamped(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN,
        encoder_hidden_dim=N_HIDDEN * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
)
mlp_clamped_losses, _ = mlp_clamped_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_clamped_losses[-1]:.6f}")

# ============================================================================
# Compute reconstruction losses
# ============================================================================
mlp_clamped_loss = compute_reconstruction_loss(mlp_clamped_model, eval_samples)

print("\n" + "=" * 70)
print("RECONSTRUCTION LOSS COMPARISON (with fair inductive biases)")
print("=" * 70)
print(f"TiedLinear (no activation):        {linear_loss:.6f}")
print(f"TiedLinearRelu (ReLU decoder):     {relu_loss:.6f}")
print(f"MLP (decoder_activation='relu'):   {mlp_loss:.6f}")
print(f"MLP (clamped to [0,1]):            {mlp_clamped_loss:.6f}")
print()
print(f"MLP vs TiedLinearRelu gap: {(mlp_loss - relu_loss) / relu_loss * 100:+.2f}%")
print(
    f"MLP clamped vs TiedLinearRelu gap: {(mlp_clamped_loss - relu_loss) / relu_loss * 100:+.2f}%"
)

# ============================================================================
# Compute angular variance for clamped MLP
# ============================================================================
print("\n" + "=" * 70)
print("Computing Angular Variance for clamped MLP...")
print("=" * 70)

mlp_clamped_avs = compute_all_angular_variances(mlp_clamped_model, test_samples)
mlp_clamped_valid = mlp_clamped_avs[~np.isnan(mlp_clamped_avs)]

print(f"\nAngular Variance comparison:")
print(f"  TiedLinearRelu: {np.nanmean(relu_avs):.6f}")
print(f"  MLP (relu dec): {np.nanmean(mlp_avs):.6f}")
print(f"  MLP (clamped):  {np.nanmean(mlp_clamped_avs):.6f}")

# ============================================================================
# Visualize comparison
# ============================================================================
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Training Loss Comparison",
        "Angular Variance Distribution",
        "Per-Feature Reconstruction Loss",
        "AV vs Reconstruction Improvement (Clamped MLP)",
    ],
)

# Training losses
fig.add_trace(
    go.Scatter(y=relu_losses, name="TiedLinearRelu", mode="lines"), row=1, col=1
)
fig.add_trace(go.Scatter(y=mlp_losses, name="MLP (relu)", mode="lines"), row=1, col=1)
fig.add_trace(
    go.Scatter(y=mlp_clamped_losses, name="MLP (clamped)", mode="lines"), row=1, col=1
)

# AV distributions
fig.add_trace(
    go.Box(y=relu_valid, name="TiedLinearRelu", boxpoints="outliers"), row=1, col=2
)
fig.add_trace(
    go.Box(y=mlp_valid, name="MLP (relu)", boxpoints="outliers"), row=1, col=2
)
fig.add_trace(
    go.Box(y=mlp_clamped_valid, name="MLP (clamped)", boxpoints="outliers"),
    row=1,
    col=2,
)

# Per-feature loss
mlp_clamped_per_feat = compute_per_feature_loss(mlp_clamped_model, eval_samples)
fig.add_trace(
    go.Scatter(
        y=compute_per_feature_loss(relu_model, eval_samples),
        name="TiedLinearRelu",
        mode="lines",
        opacity=0.7,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(y=mlp_per_feat, name="MLP (relu)", mode="lines", opacity=0.7),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(y=mlp_clamped_per_feat, name="MLP (clamped)", mode="lines"), row=2, col=1
)

# AV vs improvement for clamped MLP
improvement_clamped = ((linear_per_feat - mlp_clamped_per_feat) / linear_per_feat) * 100
valid_improvements_clamped = improvement_clamped[~np.isnan(mlp_clamped_avs)]
valid_avs_clamped = mlp_clamped_avs[~np.isnan(mlp_clamped_avs)]
corr_clamped, _ = stats.pearsonr(valid_avs_clamped, valid_improvements_clamped)

fig.add_trace(
    go.Scatter(
        x=valid_avs_clamped,
        y=valid_improvements_clamped,
        mode="markers",
        marker=dict(size=5, opacity=0.6),
        name=f"r={corr_clamped:.2f}",
        showlegend=True,
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_yaxes(title_text="Loss", type="log", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=2)
fig.update_xaxes(title_text="Feature Index", row=2, col=1)
fig.update_yaxes(title_text="MSE", row=2, col=1)
fig.update_xaxes(title_text="Angular Variance", row=2, col=2)
fig.update_yaxes(title_text="Reconstruction Improvement (%)", row=2, col=2)

fig.update_layout(
    height=700, title_text="MLP with Fair Inductive Biases vs TiedLinearRelu"
)
fig.show()

# ============================================================================
# Summary
# ============================================================================
print("\n" + "=" * 70)
print("FAIR COMPARISON SUMMARY")
print("=" * 70)
print(f"""
The TiedLinearRelu has an inductive bias advantage: its ReLU decoder ensures
non-negative outputs, matching the [0,1] feature distribution perfectly.

Comparison with matched biases:
  - TiedLinearRelu MSE:     {relu_loss:.6f}
  - MLP (relu decoder) MSE: {mlp_loss:.6f}  ({(mlp_loss - relu_loss) / relu_loss * 100:+.1f}%)
  - MLP (clamped [0,1]) MSE: {mlp_clamped_loss:.6f}  ({(mlp_clamped_loss - relu_loss) / relu_loss * 100:+.1f}%)

Angular Variance (manifold structure):
  - TiedLinearRelu:  {np.nanmean(relu_avs):.6f} (piecewise linear = no manifold)
  - MLP (relu dec):  {np.nanmean(mlp_avs):.6f} ({np.nanmean(mlp_avs) / max(np.nanmean(relu_avs), 1e-10):.0f}x)
  - MLP (clamped):   {np.nanmean(mlp_clamped_avs):.6f} ({np.nanmean(mlp_clamped_avs) / max(np.nanmean(relu_avs), 1e-10):.0f}x)

Key insight: Even with matched inductive biases (clamped output), the MLP shows
{"SIGNIFICANT" if np.nanmean(mlp_clamped_avs) > 0.01 else "minimal"} manifold structure (AV > 0), 
while achieving {"comparable" if abs(mlp_clamped_loss - relu_loss) / relu_loss < 0.1 else "different"} reconstruction quality.
""")

## 4.1 SAE on Nonlinear Representations

Train a standard SAE on the bottleneck activations of the MLP autoencoder. Compare feature recovery metrics against the linear baseline. If the activation space is curved, SAE (which assumes linear features) should show degraded performance ﷿﷿﷿ potentially new failure modes beyond absorption/splitting.

In [ ]:
from occhio.sae.sae import SAESimple

# ============================================================================
# SAE on MLP Bottleneck Activations
# ============================================================================

# Generate bottleneck activations from the trained MLP autoencoder
print("Generating MLP bottleneck activations...")
n_sae_samples = 50000
sae_train_inputs = dist.sample(n_sae_samples)

with torch.no_grad():
    mlp_bottleneck = mlp_model.ae.encode(
        sae_train_inputs
    )  # Shape: (n_samples, N_HIDDEN)

print(f"Bottleneck shape: {mlp_bottleneck.shape}")
print(
    f"Bottleneck stats: min={mlp_bottleneck.min():.3f}, max={mlp_bottleneck.max():.3f}, mean={mlp_bottleneck.mean():.3f}"
)

# Similarly for linear baseline
with torch.no_grad():
    linear_bottleneck = linear_model.ae.encode(sae_train_inputs)

print(f"Linear bottleneck shape: {linear_bottleneck.shape}")

In [ ]:
# Train SAEs on both bottleneck spaces
# Using dictionary size = N_FEATURES to attempt feature recovery

N_SAE_STEPS = 5000
SAE_L1_COEF = 0.05

print("=" * 70)
print("Training SAE on MLP bottleneck activations...")
print("=" * 70)

sae_mlp = SAESimple(
    n_latent=N_HIDDEN,
    n_dict=N_FEATURES,
    l1_coef=SAE_L1_COEF,
)

# Data function for SAE training
mlp_idx = 0


def mlp_data_fn(batch_size):
    global mlp_idx
    start = mlp_idx % len(mlp_bottleneck)
    end = start + batch_size
    if end > len(mlp_bottleneck):
        # Wrap around
        batch = torch.cat(
            [mlp_bottleneck[start:], mlp_bottleneck[: end - len(mlp_bottleneck)]]
        )
    else:
        batch = mlp_bottleneck[start:end]
    mlp_idx += batch_size
    return batch


sae_mlp_losses = sae_mlp.train_sae(mlp_data_fn, n_steps=N_SAE_STEPS, batch_size=512)

print("\n" + "=" * 70)
print("Training SAE on Linear bottleneck activations...")
print("=" * 70)

sae_linear = SAESimple(
    n_latent=N_HIDDEN,
    n_dict=N_FEATURES,
    l1_coef=SAE_L1_COEF,
)

linear_idx = 0


def linear_data_fn(batch_size):
    global linear_idx
    start = linear_idx % len(linear_bottleneck)
    end = start + batch_size
    if end > len(linear_bottleneck):
        batch = torch.cat(
            [
                linear_bottleneck[start:],
                linear_bottleneck[: end - len(linear_bottleneck)],
            ]
        )
    else:
        batch = linear_bottleneck[start:end]
    linear_idx += batch_size
    return batch


sae_linear_losses = sae_linear.train_sae(
    linear_data_fn, n_steps=N_SAE_STEPS, batch_size=512
)

In [ ]:
# ============================================================================
# Feature Recovery Metrics
# ============================================================================
# For each true feature, find the SAE dictionary element with highest correlation
# to its activation pattern

print("=" * 70)
print("FEATURE RECOVERY ANALYSIS")
print("=" * 70)

# Generate test data for evaluation
n_test = 10000
test_inputs = dist.sample(n_test)

with torch.no_grad():
    # Get bottleneck activations
    mlp_test_bottleneck = mlp_model.ae.encode(test_inputs)
    linear_test_bottleneck = linear_model.ae.encode(test_inputs)

    # Get SAE activations
    _, sae_mlp_acts = sae_mlp(mlp_test_bottleneck)
    _, sae_linear_acts = sae_linear(linear_test_bottleneck)


# Compute correlation between true features and SAE dictionary elements
def compute_feature_recovery(true_features, sae_activations):
    """
    For each true feature, find the max correlation with any SAE dictionary element.
    Returns: max_correlations (per feature), best_matches (indices)
    """
    n_features = true_features.shape[1]
    n_dict = sae_activations.shape[1]

    max_corrs = []
    best_matches = []

    for feat_idx in range(n_features):
        feat_vals = true_features[:, feat_idx].numpy()
        if feat_vals.std() < 1e-8:
            max_corrs.append(0.0)
            best_matches.append(-1)
            continue

        best_corr = 0.0
        best_idx = -1

        for dict_idx in range(n_dict):
            dict_vals = sae_activations[:, dict_idx].numpy()
            if dict_vals.std() < 1e-8:
                continue
            corr = np.corrcoef(feat_vals, dict_vals)[0, 1]
            if not np.isnan(corr) and corr > best_corr:
                best_corr = corr
                best_idx = dict_idx

        max_corrs.append(best_corr)
        best_matches.append(best_idx)

    return np.array(max_corrs), np.array(best_matches)


print("\nComputing feature recovery for MLP SAE...")
mlp_recovery, mlp_matches = compute_feature_recovery(test_inputs, sae_mlp_acts)

print("Computing feature recovery for Linear SAE...")
linear_recovery, linear_matches = compute_feature_recovery(test_inputs, sae_linear_acts)

print(f"\n{'Metric':<40} {'MLP SAE':>12} {'Linear SAE':>12}")
print("-" * 65)
print(
    f"{'Mean feature recovery (correlation)':<40} {mlp_recovery.mean():>12.4f} {linear_recovery.mean():>12.4f}"
)
print(
    f"{'Median feature recovery':<40} {np.median(mlp_recovery):>12.4f} {np.median(linear_recovery):>12.4f}"
)
print(
    f"{'Features with corr > 0.5':<40} {(mlp_recovery > 0.5).sum():>12d} {(linear_recovery > 0.5).sum():>12d}"
)
print(
    f"{'Features with corr > 0.7':<40} {(mlp_recovery > 0.7).sum():>12d} {(linear_recovery > 0.7).sum():>12d}"
)
print(
    f"{'Features with corr > 0.9':<40} {(mlp_recovery > 0.9).sum():>12d} {(linear_recovery > 0.9).sum():>12d}"
)

# Check for dead dictionary elements
mlp_dead = (sae_mlp_acts.sum(0) == 0).sum().item()
linear_dead = (sae_linear_acts.sum(0) == 0).sum().item()
print(f"{'Dead dictionary elements':<40} {mlp_dead:>12d} {linear_dead:>12d}")

In [ ]:
# ============================================================================
# Visualization: Feature Recovery vs Angular Variance
# ============================================================================

fig = go.Figure()

# Scatter plot: Recovery vs Angular Variance for each feature
fig.add_trace(
    go.Scatter(
        x=mlp_avs[: len(mlp_recovery)],
        y=mlp_recovery,
        mode="markers",
        marker=dict(
            size=6,
            color=IMPORTANCES.numpy()[: len(mlp_recovery)],
            colorscale="Viridis",
            colorbar=dict(title="Feature<br>Importance"),
            opacity=0.6,
        ),
        text=[f"Feature {i}" for i in range(len(mlp_recovery))],
        hovertemplate="Feature %{text}<br>Angular Variance: %{x:.4f}<br>Recovery: %{y:.4f}<extra></extra>",
        showlegend=False,
    )
)

# Add trendline
valid_mask = ~np.isnan(mlp_avs[: len(mlp_recovery)])
valid_avs_for_plot = mlp_avs[: len(mlp_recovery)][valid_mask]
valid_recovery_for_plot = mlp_recovery[valid_mask]

if len(valid_avs_for_plot) > 0:
    z = np.polyfit(valid_avs_for_plot, valid_recovery_for_plot, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(valid_avs_for_plot.min(), valid_avs_for_plot.max(), 100)

    corr_av_rec = np.corrcoef(valid_avs_for_plot, valid_recovery_for_plot)[0, 1]

    fig.add_trace(
        go.Scatter(
            x=x_trend,
            y=p(x_trend),
            mode="lines",
            line=dict(dash="dash", color="red", width=2),
            name=f"Trend (r={corr_av_rec:.3f})",
            showlegend=True,
        )
    )

fig.update_layout(
    title="SAE Feature Recovery vs Angular Variance (MLP)",
    xaxis_title="Angular Variance",
    yaxis_title="Recovery (Correlation)",
    height=500,
    width=800,
    hovermode="closest",
)

fig.show()

print(f"\nCorrelation between Angular Variance and Recovery: {corr_av_rec:.4f}")
print(f"Features analyzed: {len(valid_recovery_for_plot)}/{len(mlp_recovery)}")

In [ ]:
# ============================================================================
# Key Question: Does manifold structure cause SAE failure modes?
# ============================================================================

print("=" * 70)
print("SAE FAILURE MODE ANALYSIS")
print("=" * 70)

# Hypothesis: Features with high angular variance (curved encoding) should have
# worse SAE recovery because SAE assumes linear features

# Correlation between angular variance and recovery
av_recovery_corr = np.corrcoef(mlp_avs[: len(mlp_recovery)], mlp_recovery)[0, 1]
print(
    f"\nCorrelation between Angular Variance and SAE Recovery: {av_recovery_corr:.4f}"
)

# Split features by angular variance
av_median = np.median(mlp_avs)
high_av_mask = mlp_avs[: len(mlp_recovery)] > av_median
low_av_mask = ~high_av_mask

print(f"\nRecovery by Angular Variance group:")
print(
    f"  High AV features (>{av_median:.4f}): mean recovery = {mlp_recovery[high_av_mask].mean():.4f}"
)
print(
    f"  Low AV features  (<={av_median:.4f}): mean recovery = {mlp_recovery[low_av_mask].mean():.4f}"
)

# Statistical test
from scipy.stats import mannwhitneyu

stat, p_val_av = mannwhitneyu(
    mlp_recovery[high_av_mask], mlp_recovery[low_av_mask], alternative="less"
)
print(f"  Mann-Whitney U test (high < low): p = {p_val_av:.4f}")

# Check for absorption: multiple true features mapping to same SAE element
mlp_unique_matches = len(set(mlp_matches[mlp_matches >= 0]))
linear_unique_matches = len(set(linear_matches[linear_matches >= 0]))

print(f"\nAbsorption check (multiple features -> same SAE element):")
print(f"  MLP SAE: {N_FEATURES} features -> {mlp_unique_matches} unique SAE elements")
print(
    f"  Linear SAE: {N_FEATURES} features -> {linear_unique_matches} unique SAE elements"
)

# Features per SAE element
from collections import Counter

mlp_match_counts = Counter(mlp_matches[mlp_matches >= 0])
linear_match_counts = Counter(linear_matches[linear_matches >= 0])

mlp_absorbed = sum(1 for c in mlp_match_counts.values() if c > 1)
linear_absorbed = sum(1 for c in linear_match_counts.values() if c > 1)

print(f"  MLP: {mlp_absorbed} SAE elements absorbing >1 feature")
print(f"  Linear: {linear_absorbed} SAE elements absorbing >1 feature")

print("\n" + "=" * 70)
print("SUMMARY: SAE on Nonlinear Representations")
print("=" * 70)
print(f"""
If angular variance correlates negatively with SAE recovery, this suggests
that manifold structure (curved encoding space) creates a fundamental 
incompatibility with SAE's linear feature assumption.

Key findings:
- AV-Recovery correlation: {av_recovery_corr:.4f} {"(negative = manifold hurts SAE)" if av_recovery_corr < 0 else "(positive = unexpected)"}
- MLP SAE mean recovery: {mlp_recovery.mean():.4f}
- Linear SAE mean recovery: {linear_recovery.mean():.4f}
- Recovery difference: {linear_recovery.mean() - mlp_recovery.mean():.4f} {"(linear better)" if linear_recovery.mean() > mlp_recovery.mean() else "(MLP better)"}
""")